# Content assertions and successors

Open this `.ipynb` in JupyterLab or an editor with Jupyter support. Install Deno's kernel with
`deno jupyter --install`, select **Deno**, and Run All from a fresh kernel. This notebook imports
exact published verifier **2.0.0**. It generates its own demonstration keys, asserts success and
refusal, prints only safe semantic outcomes and cleans up resources. The saved text is the expected
semantic output; CI executes each cell in the real kernel and compares it. Random key and signature
bytes are never persisted. See [runner instructions](README.md).

Verification is not authority. The host owns trusted inputs, replay persistence, revocation and
policy. The HTTP demonstration's memory reservations last one process; use the durable-store
recipe for a persistent host.


In [ ]:
import assertPackage from "node:assert/strict";
import verifierPackage from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
import { BACKEND_ID as notebookBackend } from "npm:@bounded-authority-protocol/verifier@2.0.0";
assertPackage.equal(verifierPackage.version, "2.0.0");
assertPackage.equal(notebookBackend, "portable");
console.log("verifier 2.0.0; portable backend");


verifier 2.0.0; portable backend


In [ ]:
import assert from "node:assert/strict";
import { createHash, generateKeyPairSync, sign, type KeyObject } from "node:crypto";
import { boundsNew, contentAssertion, strUtf8 } from "npm:@bounded-authority-protocol/verifier@2.0.0";

const raw = (key: KeyObject): Uint8Array => Buffer.from(key.export({ format: "jwk" }).x!, "base64url");
const publisher = generateKeyPairSync("ed25519");
const attestor = { keyId: "publisher-1", publicKey: raw(publisher.publicKey), validFrom: 0, validBefore: null };
const profileDigest = createHash("sha256").update("retention-policy schema 3").digest();
const bounds = boundsNew();

function publish(text: string, gen: number, prev: Uint8Array, iat: number) {
  const contentDigest = contentAssertion.contentDigest(strUtf8(text));
  assert.ok(contentDigest.ok);
  const input = contentAssertion.assertionSigningInput({
    attestorKeyId: "publisher-1", jti: `retention-policy-${gen}`, iss: "https://records.example",
    aud: "https://archive.example", sub: "policy:retention", profile: "urn:example:retention:3",
    profileDigest, contentDigest: contentDigest.value, gen, prev, iat, nbf: iat, exp: iat + 86_400,
  });
  assert.ok(input.ok);
  const message = Buffer.concat([input.value.protectedSegment, Buffer.from("."), input.value.payloadSegment]);
  const compact = contentAssertion.assembleCompact(input.value, sign(null, message, publisher.privateKey));
  assert.ok(compact.ok);
  const facts = contentAssertion.verifyAssertion(compact.value, {
    attestor, issuer: "https://records.example", audience: "https://archive.example", subject: "policy:retention",
    profile: "urn:example:retention:3", profileDigest, contentDigest: contentDigest.value, now: iat + 10, bounds,
  });
  assert.ok(facts.ok);
  return facts.value;
}

const first = publish("keep invoices 7 years", 1, new Uint8Array(32), 1_000);
const second = publish("keep invoices 10 years", 2, first.digest, 2_000);
assert.ok(contentAssertion.verifySuccessor(first, second, bounds).ok);

// Skipping a generation, or naming the wrong predecessor, rejects.
const third = publish("keep invoices 5 years", 3, second.digest, 3_000);
assert.deepEqual(contentAssertion.verifySuccessor(first, third, bounds), { ok: false });
const fork = publish("keep invoices 1 year", 2, new Uint8Array(32).fill(9), 2_500);
assert.deepEqual(contentAssertion.verifySuccessor(first, fork, bounds), { ok: false });

console.log("content: successor verified; skipped generation and fork refused");


content: successor verified; skipped generation and fork refused
